# Multi-Replica Routing — the correlation knob

**One arrival stream. N GPUs, each a full copy of the model. Who gets what?**

That decision is *multi-replica routing* — Vidur's `global_scheduler` — and until now it
was the one L0/L1 feature this simulator did not have. It is easy to file it under
"load balancing, a latency concern". It is not:

> **Facility peak depends on whether the replicas are busy at the same time, and the
> router is the only component in the stack that decides that.**

Everything else here prices work. This layer decides how work **coincides** — which is
the quantity that sizes a breaker.

### What this notebook does

| § | question |
|---|---|
| 1 | Setup — traffic, templates, predictor |
| 2 | The four policies, and what each can *see* |
| 3 | One fleet, end to end: the facility trace |
| 4 | Same traffic, four routers — the comparison the layer exists for |
| 5 | **Load sweep** — the policy only matters when the fleet is crowded |
| 6 | Why Vidur's `lor` collapses to a hot spot, from its source |
| 7 | The aperture problem, at fleet scale |
| 8 | What this cannot tell you |

### Read this first

Every number below comes from the **analytic roofline** backend unless you supply the
measured LUT in §1. The roofline is an *ansatz* — `max(flops/peak, bytes/bandwidth)` —
and it gets trends right **by construction**, which is exactly why it can never falsify
anything. Figures are stamped `SYNTHETIC` when that is what produced them. Routing
conclusions about *relative* behaviour survive the swap; absolute watts do not.

---
## 1. Setup

On Colab, clone the repo. Locally, just run from the repo root.

**The routing layer is newer than the rest of the package.** `dynshape/routing.py`,
`fleet.py` and `fleet_plot.py` did not exist before this notebook did, so a checkout that
predates them fails every import below with an opaque
`ImportError: cannot import name 'FleetConfig'`. The cell checks for them by name and says
so directly — set `BRANCH` to whichever branch carries the routing commit.

In [ ]:
import os, sys

REPO   = "dynamic_shape_power_sim"
GITHUB = "https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git"
BRANCH = "main"          # <- the branch carrying the routing layer

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.isdir(REPO):
            !git clone -q --branch $BRANCH $GITHUB
        os.chdir(REPO)
    !pip -q install numpy pandas matplotlib

sys.path.insert(0, os.getcwd())

# Fail on the real problem rather than on a symptom five lines later.
NEEDED = ["dynshape/routing.py", "dynshape/fleet.py", "dynshape/fleet_plot.py"]
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    raise RuntimeError(
        "This checkout has no multi-replica routing layer.\n"
        f"  missing : {', '.join(missing)}\n"
        f"  cwd     : {os.getcwd()}\n"
        f"  branch  : {BRANCH}\n\n"
        "Commit and push routing.py / fleet.py / fleet_plot.py (and the __init__.py that\n"
        "exports them) to that branch, or run this notebook from a working tree that has\n"
        "them. Nothing below can work until dynshape exports FleetConfig.")
print("routing layer present in", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams["figure.dpi"] = 110

In [ ]:
from dynshape import (ShapeRewriter, build_predictor, generate_traffic, TrafficConfig,
                      traffic_summary, EngineConfig, SchedulerConfig,
                      FleetConfig, run_fleet, compare_routing, clone_requests,
                      build_router, assign_static, routing_balance, ROUTING_POLICIES)
from dynshape.fleet_plot import (plot_fleet_dashboard, plot_facility_power,
                                 plot_routing_comparison, plot_replica_balance,
                                 plot_replica_duty, plot_aperture_sensitivity)

rewriter = ShapeRewriter.from_dir("templates/gpt2")

# force_analytic=True is the roofline. To use the measured EnergAIzer LUT instead:
#     from dynshape import build_gee_predictor, download_lut
#     predictor = build_gee_predictor(...)
# and note that changing the predictor changes the *schedule* -- it drives the clock,
# so a roofline run and a measured run on identical traffic are close cousins, not twins.
predictor = build_predictor(force_analytic=True)
print(predictor.stats()["backend"])

### The traffic

One stream, and every router below sees the identical stream — same arrival times, same
prompt lengths, same output lengths, down to the token. That is not a courtesy; it is
what makes the differences attributable.

`max_tokens=1024` because **GPT-2's context window is 1024 tokens** and the rewriter is
pure arithmetic — it will happily price a 2048-token GPT-2 context that the model cannot
run (design doc, assumption 10).

In [ ]:
MAX_TOKENS = 1024
N_REPLICAS = 4

traffic_cfg = TrafficConfig(interval="poisson", qps=120, length="zipf",
                            num_requests=300, max_tokens=MAX_TOKENS, seed=0)
requests = generate_traffic(traffic_cfg)

engine_cfg = EngineConfig(scheduler=SchedulerConfig(max_tokens=MAX_TOKENS),
                          record_kernels_until_ms=100.0)

pd.Series(traffic_summary(requests)).to_frame("value")

---
## 2. The policies, and what each one can *see*

A router is a pure function of `(arrival order, replica load)`. What separates the
policies is **how much of the load they are allowed to look at**.

| policy | rule | source |
|---|---|---|
| `round_robin` | take turns in order | Vidur, exact |
| `random` | uniform over replicas | Vidur, reseeded onto its own stream |
| `lor` | fewest **queued** requests | Vidur, exact |
| `lor_requests` | fewest queued **+ running** | ours |
| `lor_tokens` | fewest outstanding *tokens* | ours |
| `lor_kv` | lowest KV utilisation | ours |

The `reactive` flag is the real distinction. A non-reactive router's assignment can be
computed **before any simulation runs**, because it never looks at a replica. A reactive
one only exists while the fleet runs — which is precisely why the fleet loop is
event-driven.

In [ ]:
from dynshape.routing import ReplicaLoad

rows = []
for name in ROUTING_POLICIES:
    r = build_router(name, N_REPLICAS, seed=0)
    rows.append({"policy": name, "reactive": r.reactive,
                 "assignment known in advance": not r.reactive})
pd.DataFrame(rows)

### One fleet state, six answers

Replica 0 holds many small requests; replica 1 holds one enormous prefill; replica 2 is
moderately loaded but its KV pool is nearly full. Where should the next request go?

In [ ]:
state = [
    ReplicaLoad(0, queued_requests=6, running_requests=2, pending_tokens=600,  kv_utilisation=0.10),
    ReplicaLoad(1, queued_requests=1, running_requests=1, pending_tokens=8000, kv_utilisation=0.35),
    ReplicaLoad(2, queued_requests=3, running_requests=9, pending_tokens=3000, kv_utilisation=0.95),
    ReplicaLoad(3, queued_requests=4, running_requests=4, pending_tokens=2000, kv_utilisation=0.50),
]
probe = requests[0]

pd.DataFrame([{"policy": p, "picks replica": build_router(p, 4, seed=0).route(probe, state)}
              for p in ROUTING_POLICIES])

They disagree, and each is defensible:

- `lor` picks **1** — one queued request looks emptiest, never mind the 8000 tokens behind it.
- `lor_tokens` picks **0** — least actual work outstanding.
- `lor_kv` picks **0** — most room in the resource that actually runs out.

> **`lor` counting requests rather than work is Vidur's own caution**, and its comment says
> so: *"using a very simple implementation here, to keep wiring simple."* §6 shows what it
> costs.

### Counts are not load

`round_robin` splits request *counts* exactly. That is not the same as splitting work,
and nothing in it looks at how big a request is. Six requests, two of them giants,
landing three apart:

In [ ]:
from dynshape.entities import SimRequest, reset_ids

reset_ids()
demo = [SimRequest(arrived_at=0.01 * i, num_prefill_tokens=n, num_decode_tokens=10)
        for i, n in enumerate([4000, 50, 50, 4000, 50, 50])]
bal = routing_balance(assign_static(demo, build_router("round_robin", 3), 3), demo, 3)

print("requests per replica:", bal["per_replica_requests"], " max/mean",
      f"{bal['requests']['max_over_mean']:.2f}   <- perfect")
print("tokens   per replica:", bal["per_replica_tokens"], " max/mean",
      f"{bal['tokens']['max_over_mean']:.2f}   <- not perfect")

Both giants land on replica 0 purely because they arrived three apart. Not contrived —
request sizes and arrival order are uncorrelated, so that alignment happens by chance.

---
## 3. One fleet, end to end

Four replicas, round robin, the traffic from §1. This is the whole layer working:
one arrival stream in, four independent power traces out, and a facility total that is
**not** four times a single GPU.

In [ ]:
fleet = run_fleet(clone_requests(requests), rewriter, predictor,
                  FleetConfig(num_replicas=N_REPLICAS, routing="round_robin",
                              engine=engine_cfg))
pd.Series(fleet.summary(dt_ms=250.0)).to_frame("value")

In [ ]:
fig = plot_fleet_dashboard(fleet, dt_ms=250.0)
plt.show()

### How to read the stacked panel

The **height of the stack is the facility draw** — that is why it is stacked and not
overlaid. Three things to notice:

1. **The dotted line at `N × 47.35 W`** is the idle floor. Every GPU draws it whether or
   not it is doing anything, and no router can move it. Only the part above it is in play.
2. **The red dashed line is the facility peak at a 250 ms aperture.** Change the aperture
   and that number changes (§7).
3. `tail_idle_energy_j` in the summary is what the *early finishers* burned waiting for the
   last replica. Charged explicitly, because dropping it is how a fleet report quietly
   understates the floor.

---
## 4. Same traffic, four routers

The experiment this layer exists for. Every policy gets a **clone** of the request list,
so arrivals, prompt lengths and output lengths are identical to the token — the only
thing that varies is which replica each request landed on.

In [ ]:
POLICIES = ("round_robin", "random", "lor", "lor_requests", "lor_tokens")

traces, table = compare_routing(requests, rewriter, predictor,
                                policies=POLICIES,
                                config=FleetConfig(num_replicas=N_REPLICAS,
                                                   engine=engine_cfg),
                                dt_ms=250.0, progress=True)

table[["routing", "wall_time_s", "fleet_duty_cycle", "total_energy_j",
       "facility_peak_w", "dynamic_peak_w", "coincidence_factor",
       "dynamic_coincidence_factor", "tokens_max_over_mean",
       "ttft_p99_s"]].set_index("routing")

In [ ]:
fig = plot_routing_comparison(traces, dt_ms=250.0)
plt.show()

### The coincidence factor, and why you must read the *dynamic* one

$$\text{coincidence} = \frac{\text{facility peak}}{\sum_k \text{replica}_k \text{ peak}}$$

**1.0** means every replica peaked in the same bin — no smoothing at all. **1/N** means
they never did.

The raw factor is close to 1.0 for every policy, and that is not a result about routers.
Each replica draws ~47 W whatever it is doing, so on any fleet below full duty cycle that
constant sits in the numerator *and* the denominator and drags the ratio toward 1. The
**dynamic** factor subtracts `N × P_idle` first and measures what actually varies.

> This is not a simulator artefact. It is what a facility meter sees, and it is why
> `facility_floor_w` is reported as its own line. At low duty cycle **most of a fleet's
> bill is the floor**, and no scheduling decision touches it.

In [ ]:
fig, axes = plt.subplots(1, len(traces), figsize=(3.4 * len(traces), 3.6), sharey=True)
for ax, (name, tr) in zip(np.atleast_1d(axes), traces.items()):
    plot_replica_balance(tr, ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("")
axes[0].set_ylabel("share, relative to an even split")
fig.tight_layout()
plt.show()

---
## 5. Load sweep — the policy only matters when the fleet is crowded

> *"At one request every ten seconds the elevator is always waiting and empty; whoever
> arrives rides alone, and all five policies produce identical output because there is
> never a choice to make."*

The same is true one level up. Below saturation every replica is empty when a request
arrives, all reactive routers tie, and **the tie rule becomes the policy**. Sweep the load
before concluding anything about routers.

In [ ]:
SWEEP_QPS = [30, 120, 400, 1000]
SWEEP_POLICIES = ("round_robin", "random", "lor", "lor_tokens")

sweep_rows = []
for qps in SWEEP_QPS:
    reqs_q = generate_traffic(TrafficConfig(interval="poisson", qps=qps, length="zipf",
                                            num_requests=200, max_tokens=MAX_TOKENS,
                                            seed=0))
    for pol in SWEEP_POLICIES:
        tr = run_fleet(clone_requests(reqs_q), rewriter, predictor,
                       FleetConfig(num_replicas=N_REPLICAS, routing=pol,
                                   engine=engine_cfg))
        s = tr.summary(dt_ms=250.0)
        sweep_rows.append({"qps": qps, "routing": pol,
                           "duty": s["fleet_duty_cycle"],
                           "facility_peak_w": s["facility_peak_w"],
                           "dynamic_peak_w": s["dynamic_peak_w"],
                           "dyn_coincidence": s["dynamic_coincidence_factor"],
                           "tokens_imbalance": s["tokens_max_over_mean"],
                           "ttft_p99_s": s["ttft_p99_s"],
                           "energy_per_tok_mj": s["energy_per_output_token_mj"],
                           "split": tr.balance()["per_replica_requests"]})
    print(f"qps={qps} done", flush=True)

sweep = pd.DataFrame(sweep_rows)
sweep.pivot(index="qps", columns="routing", values="tokens_imbalance")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for metric, ax, title in zip(
        ["facility_peak_w", "tokens_imbalance", "dyn_coincidence"], axes,
        ["facility peak (W)", "token imbalance (max/mean)",
         "dynamic coincidence factor"]):
    for pol in SWEEP_POLICIES:
        sub = sweep[sweep.routing == pol]
        ax.plot(sub.qps, sub[metric], "o-", label=pol)
    ax.set_xscale("log")
    ax.set_xlabel("arrival rate (qps)")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25, which="both")
axes[1].axhline(1.0, color="black", ls="--", lw=0.9)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

sweep[["qps", "routing", "duty", "split", "facility_peak_w", "ttft_p99_s"]]

### What the sweep says

Three things, and the middle one is the trap.

**1. `round_robin` and `random` are flat across load.** They cannot see load, so of course
they are. Their token imbalance sits near 1.0 everywhere — a property of the arrival
process, not of the router.

**2. `lor` is badly skewed at every load below saturation, and it *looks like a win*.**
It often reports the **lowest facility peak** in the table. That is not smoothing. It is
concentration: work piles onto replica 0, the others sit idle, the run takes longer, and
the same energy spread over more time is a lower peak. Read `duty` and `split` beside it —
a peak lowered by leaving hardware unused is a way of failing, not a way of winning. The
coincidence factor cannot distinguish the two on its own, which is why `balance()` and
`peak_to_mean` are reported next to it.

**3. At saturation everything converges.** Every replica is busy in every bin, the
dynamic coincidence factor goes to 1.0, and no router can help. **The routing choice
moves facility peak most in the middle** — busy enough that replicas can be out of phase,
not so busy that they are all pinned.

On the roofline backend, this configuration gives the pattern below. Absolute watts are
synthetic; the shape is the claim. Read the **spread** row — how far apart the policies
are — not the levels:

| qps | duty (RR) | peak spread across policies | `lor` split |
|---|---|---|---|
| 30 | 0.37 | 21 W | `[195, 5, 0, 0]` |
| 120 | 0.90 | **37 W** | `[166, 31, 3, 0]` |
| 400 | 0.95 | 1 W | `[89, 70, 34, 7]` |
| 1000 | 0.94 | 3 W | `[60, 56, 47, 37]` |

The spread peaks in the middle and collapses at both ends. Note also that `random`
decorrelates slightly *better* than `round_robin` at low load (dynamic coincidence 0.60
vs 0.73 at 30 qps) — deterministic spreading is not the same as independence, and
round robin's regularity can keep replicas mildly in phase. That one is an observation
from a single seed, not a result; re-roll `traffic_cfg.seed` before believing it.

---
## 6. Why Vidur's `lor` collapses — read from the source

This is worth a section because a `[196, 4, 0, 0]` split looks exactly like a routing bug,
and it is not one. It is what the code does.

```python
# vidur/scheduler/global_scheduler/lor_global_scheduler.py
pending_requests_map = {
    replica_scheduler.replica_id: replica_scheduler.num_pending_requests
    for replica_scheduler in self._replica_schedulers.values()
}
replica_id = min(pending_requests_map.items(), key=lambda x: x[1])[0]

# vidur/scheduler/replica_scheduler/base_replica_scheduler.py
@property
def num_pending_requests(self) -> int:
    return len(self._request_queue)       # <- the WAITING queue only
```

`num_pending_requests` counts requests that have arrived and have **not yet been admitted**.
A replica with a hundred requests mid-decode and an empty queue reports **zero**.

So whenever queues drain each iteration — i.e. whenever the fleet is not saturated —
every replica reports 0, `min()` returns the first key, and every request goes to
replica 0. **The tie rule is the policy.**

In [ ]:
busy_but_unqueued = [
    ReplicaLoad(0, queued_requests=0, running_requests=100, pending_tokens=50_000, kv_utilisation=0.90),
    ReplicaLoad(1, queued_requests=0, running_requests=0,   pending_tokens=0,      kv_utilisation=0.0),
    ReplicaLoad(2, queued_requests=0, running_requests=0,   pending_tokens=0,      kv_utilisation=0.0),
]
for pol in ("lor", "lor_requests", "lor_tokens", "lor_kv"):
    print(f"{pol:14s} -> replica {build_router(pol, 3).route(probe, busy_but_unqueued)}")

Replica 0 is saturated and the other two are empty; Vidur's `lor` sends the request to
replica 0. `lor_requests` — the same policy counting running requests too, which is what
most people mean by LOR — sends it to replica 1.

Both are in the library, `lor` stays the Vidur-faithful default so the behaviour is
reproducible, and both are pinned by tests
(`test_routing.py::test_vidur_lor_collapses_when_queues_are_empty`).

---
## 7. The aperture problem, at fleet scale

> **A peak is not a number until you say over what window.**

A breaker responds over milliseconds; a PDU reading averages over seconds. The same trace
gives very different peaks and the *same* energy — energy invariance is the control that
says the resampler is not inventing anything.

In [ ]:
ap = pd.DataFrame(fleet.aperture_table([5, 25, 100, 250, 1000, 5000]))
display(ap.set_index("aperture_ms"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_aperture_sensitivity(fleet, apertures_ms=[5, 25, 100, 250, 1000, 5000], ax=axes[0])
for dt, colour in [(25.0, "#CC0000"), (250.0, "#4477AA"), (1000.0, "#228833")]:
    t, _, total = fleet.resample(dt_ms=dt)
    axes[1].plot(t / 1000.0, total, lw=1.0, color=colour, label=f"{dt:.0f} ms")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("facility power (W)")
axes[1].set_title("the same trace, three apertures", fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25); axes[1].margins(x=0)
fig.tight_layout()
plt.show()

Energy is constant to several significant figures across every row; the peak is not.
Quoting a fleet peak without its aperture leaves that entire range of ambiguity in the
number.

---
## 8. What this cannot tell you

Stated plainly, because a fleet figure is persuasive in a way that outruns its evidence.

1. **Nothing here has been checked against real hardware.** Step 7 of the build order is
   still the only step that can say whether any of this is right; everything else,
   including this notebook, is internal consistency. See `notebooks/vLLM_Validation_Colab.ipynb`.

2. **86% of predicted energy sits in decode attention**, which this simulator models as
   per-request eager rectangles and every real engine runs as one fused paged kernel.
   That gap is unchanged by routing — it is under every replica equally — so it moves the
   *level* of every line in every figure above, and largely not their *relative* order.

3. **Replicas are identical, and requests never migrate.** No heterogeneous fleets, no
   model-aware routing, and — the interesting omission — **no affinity for prefix-cache
   hits**. Returning a follow-up turn to the replica already holding its KV is a large
   win real systems increasingly chase, and neither Vidur nor FSTS models it. With
   `cached_prefix_tokens` already on `SimRequest`, that is the natural next router.

4. **The router is free and instantaneous.** No queueing at the balancer, no network.

5. **Single-GPU replicas.** Tensor and pipeline parallelism remain out of scope: this is
   N independent model copies, not one model sharded across N devices.

### The claim that survives

A load balancer chosen for latency reasons quietly sets facility peak power — and the
size of that effect depends on load in a non-monotonic way: **negligible when the fleet is
idle, largest in the middle, negligible again at saturation.** Nobody has reported it,
FSTS has no routing layer to test it with, and Vidur ships all three policies ready to
sweep.